# d_fit

Run all cells. Outputs are written to this task's `output/` folder.


In [6]:
%run ~/Desktop/SHL_dblp_comparable/common/core.ipynb


/Users/slmagid/miniforge3/envs/research313/lib/python3.13/site-packages/nbformat/__init__.py:96: MissingIDFieldWarning: Cell is missing an id field, this will become a hard error in future nbformat versions. You may want to use `normalize()` on your notebooks before validations (available since nbformat 5.1.4). Previous versions of nbformat are fixing this issue transparently, and will stop doing so in the future.
  validate(nb)


In [7]:
from pathlib import Path
import json, shutil
import numpy as np
import pandas as pd
ROOT = Path.home() / 'Desktop' / 'SHL_dblp_comparable'
CONFIG = read_config(ROOT)
UP = ROOT / 'c_prepare' / 'output'
OUTPUT = ROOT / 'd_fit' / 'output'
MODELS = OUTPUT / 'models'
RESULTS = OUTPUT / 'results'
for p in (MODELS, RESULTS):
    if p.exists():
        shutil.rmtree(p)
    p.mkdir(parents=True, exist_ok=True)
rows = pd.read_csv(UP / 'modeling_transitions.csv.gz', low_memory=False)
groups = sorted(rows.analysis_group.unique())
aarc_root = Path.home() / 'Desktop' / 'SHL_aarc_modeling'
aarc_selected = aarc_root / 'd_fit' / 'output' / 'results' / 'selected_hyperparameters.json'
aarc_manifest = aarc_root / 'd_fit' / 'output' / 'analysis_manifest.json'
if not aarc_selected.exists() or not aarc_manifest.exists():
    raise FileNotFoundError('Run SHL_aarc_modeling through d_fit first; strict comparison reuses its selected ridge')
selected_aarc = json.loads(aarc_selected.read_text())
manifest_aarc = json.loads(aarc_manifest.read_text())
if manifest_aarc.get('max_observed_years') != 13 or manifest_aarc.get('ar_order') != 6 or manifest_aarc.get('raw_ar_lags') is not True:
    raise ValueError('AARC analysis manifest is not comparable: require 13 years, AR(6,6), and raw lag columns')
ridge_primary = float(selected_aarc['ridge_ar'])
rho_grid = np.linspace(CONFIG['rho_min'], CONFIG['rho_max'], int(CONFIG['rho_grid_points']))
ridge_grid = np.asarray(CONFIG['ridge_grid'], float)
n_boot = int(CONFIG['n_parameter_bootstrap'])
n_gap = int(CONFIG['n_nll_gap_bootstrap'])
rho_profiles = []
ridge_profiles = []
ridge_parameters = []
chosen_rho = {}
local_ridge = {}
for group in groups:
    g = rows[rows.analysis_group.eq(group)]
    tuning = g[~g.split.eq('test')]
    rho_scores = []
    for rho in rho_grid:
        h = add_t5_history(tuning, rho)
        model = fit_model(h[h.split.eq('train')], MODEL_T5, float(CONFIG['ridge_t5']))
        score = mean_nll(model, h[h.split.eq('validation')])
        rho_profiles.append({'analysis_group': group, 'rho': rho, 'validation_nll': score})
        rho_scores.append((score, rho))
    chosen_rho[group] = float(min(rho_scores)[1])
    ridge_scores = []
    for ridge in ridge_grid:
        model = fit_model(tuning[tuning.split.eq('train')], MODEL_AR, float(ridge))
        score = mean_nll(model, tuning[tuning.split.eq('validation')])
        ridge_profiles.append({'analysis_group': group, 'ridge': ridge, 'validation_nll': score})
        ridge_scores.append((score, ridge))
        table = parameter_table(model, group, MODEL_AR)
        table['ridge'] = ridge
        ridge_parameters.append(table)
    local_ridge[group] = float(min(ridge_scores)[1])
pd.DataFrame(rho_profiles).to_csv(RESULTS / 't5_rho_profile.csv', index=False)
pd.DataFrame(ridge_profiles).to_csv(RESULTS / 'dblp_local_ridge_profile.csv', index=False)
pd.concat(ridge_parameters, ignore_index=True).to_csv(RESULTS / 'ridge_sensitivity_coefficients.csv.gz', index=False, compression='gzip')
parameters = []
heldout = []
person_scores = []
gaps = []
for group in groups:
    g = rows[rows.analysis_group.eq(group)]
    dev = g[~g.split.eq('test')]
    test = g[g.split.eq('test')]
    h = add_t5_history(g, chosen_rho[group])
    hdev = h[~h.split.eq('test')]
    htest = h[h.split.eq('test')]
    models = {MODEL_T5: fit_model(hdev, MODEL_T5, float(CONFIG['ridge_t5'])), MODEL_AR: fit_model(dev, MODEL_AR, ridge_primary)}
    save_bundle(MODELS / f'{slug(group)}.pkl', {'analysis_group': group, 'models': models, 'rho_t5': chosen_rho[group], 'ridge_ar': ridge_primary, 'max_target_age': int(g.target_age.max()), 'config_fingerprint': fingerprint(CONFIG)})
    local_person = []
    for name, model in models.items():
        evaluation = htest if name == MODEL_T5 else test
        scored = score_rows(model, evaluation)
        scored['model'] = name
        parameters.append(parameter_table(model, group, name))
        heldout.append({'analysis_group': group, 'model': name, 'test_nll': float(scored.total_nll.mean()), 'test_transitions': len(scored), 'test_people': scored.person_domain_id.nunique()})
        per = scored.groupby('person_domain_id', as_index=False).agg(mean_nll=('total_nll', 'mean'), transitions=('total_nll', 'size'))
        per['analysis_group'] = group
        per['model'] = name
        local_person.append(per)
        person_scores.append(per)
    gap = {'analysis_group': group, **cluster_bootstrap_gap(pd.concat(local_person), stable_seed(group, 'nll-gap'), n_gap)}
    gap['practically_equivalent_0.01'] = gap['ci_low'] >= -CONFIG['practical_nll_tolerance'] and gap['ci_high'] <= CONFIG['practical_nll_tolerance']
    gaps.append(gap)
pd.concat(parameters).to_csv(RESULTS / 'stage_parameters.csv', index=False)
pd.DataFrame(heldout).to_csv(RESULTS / 'heldout_metrics.csv', index=False)
pd.concat(person_scores).to_csv(RESULTS / 'heldout_person_nll.csv.gz', index=False, compression='gzip')
pd.DataFrame(gaps).to_csv(RESULTS / 'paired_nll_gaps.csv', index=False)
nesting_errors = {g: t5_nesting_max_error(rows[rows.analysis_group.eq(g)], chosen_rho[g]) for g in groups}
if max(nesting_errors.values()) > 1e-10:
    raise AssertionError(f'T5 nesting failed: {nesting_errors}')
bootstrap = []
for group in groups:
    base = rows[rows.analysis_group.eq(group) & ~rows.split.eq('test')]
    for replicate in range(n_boot):
        sample = bootstrap_weights(base, np.random.default_rng(stable_seed(group, 'bootstrap', replicate)))
        bootstrap.append(parameter_table(fit_model(sample, MODEL_AR, ridge_primary), group, MODEL_AR, replicate))
pd.concat(bootstrap).to_csv(RESULTS / 'bootstrap_stage_parameters.csv.gz', index=False, compression='gzip')
selected = {'primary_ridge_from_aarc': ridge_primary, 'dblp_local_ridge_sensitivity': local_ridge, 'rho_t5_by_cohort': chosen_rho, 'rationale': 'AARC-selected ridge is frozen for the primary cross-source coefficient comparison; DBLP-local optima are sensitivity only'}
write_json(RESULTS / 'selected_hyperparameters.json', selected)
manifest = {'status': 'passed', 'max_observed_years': 13, 'ar_order': 6, 'memory_lags': [2, 3, 4, 5, 6], 'memory_stages': ['5-7', '8-11'], 'raw_ar_lags': True, 'ar_lag_preconditioning': False, 'stage_duration_averaging': False, 'primary_cohort': 'all_eligible_cropped13', 'early_age_nesting_interactions': True, 't5_nesting_max_error_by_cohort': nesting_errors, "core_sha256": notebook_code_sha256(ROOT / "common" / "core.ipynb"), 'sensitivity_cohorts': groups, 'primary_ridge': ridge_primary, 'aarc_manifest_config_fingerprint': manifest_aarc.get('config_fingerprint'), 'config_fingerprint': fingerprint(CONFIG), 'parameter_bootstrap_replicates': n_boot}
write_json(OUTPUT / 'analysis_manifest.json', manifest)
write_json(OUTPUT / 'quality_report.json', manifest)
print(json.dumps(selected, indent=2))


{
  "primary_ridge_from_aarc": 1e-06,
  "dblp_local_ridge_sensitivity": {
    "all_eligible_cropped13": 0.01,
    "all_eligible_no_aarc_overlap": 0.01,
    "complete13_cropped": 0.001,
    "legacy_complete20_cropped13": 0.001
  },
  "rho_t5_by_cohort": {
    "all_eligible_cropped13": 0.6206666666666666,
    "all_eligible_no_aarc_overlap": 0.6206666666666666,
    "complete13_cropped": 0.6206666666666666,
    "legacy_complete20_cropped13": 0.7839999999999999
  },
  "rationale": "AARC-selected ridge is frozen for the primary cross-source coefficient comparison; DBLP-local optima are sensitivity only"
}
